In [1]:
import os
import platform
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
import re

COLUNAS = [
    "PROPIETARIO", "TRAYECTO", "TRANSPORTISTA", "TRACTORA", "REMOLQUE",
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT", "PESO_BRUTO",
    "CODEUT", "ESTADO_UT", "RANGO_UT", "FCARGA", "ACTIVIDAD",
    "CODEDT", "ESTADO_DT", "REFERENCIA", "CODACT", "LOCORIGEN",
    "PROV_ORIGEN", "PAISORIGEN", "CPOSTAL", "LOCDESTINO", "PROV_DESTINO",
    "PAISDESTINO", "CPOSTAD", "KM", "FENTREGA", "ORIGEN",
    "ENTREGAR", "PROV_ENTREGAR", "PAISENTREGAR", "DESTINO", "PALETS",
    "PREFAC", "RUTA", "COBROREAL", "GESTION", "DEPART",
    "USCODE", "USUARIO", "TIPOCLIENTE", "TIPOFLUJO", "WMSCODRGT",
    "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TEMP_MERC_PED",
    "TIPOPALETA", "CAMION_TIPO", "CAMION_CAPACIDAD", "TIPO_COMBUSTIBLE",
    "KMREALES", "ALBARAN"
]

COLUNA_DATA = "FENTREGA"
COLUNA_ORIGEM = "ficheiro_origem"
NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"
REGEX_ANO = re.compile(r"^(\d{4})")

In [4]:
if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\00.DB\2026.db"
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Documents\python\000.Dados_input\inform_27")
elif platform.system() == 'Darwin':
    DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00.DB/2026.db"
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados")

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
con = sqlite3.connect(DB_PATH)
con.close()
print(f"✓ BD: {DB_PATH}")

✓ BD: C:\Users\LISARR\OneDrive - Salvesen Logística S.A\00.DB\2026.db


In [3]:
import csv

# ==========================
# Funções auxiliares
# ==========================
def obter_ano(valor):
    if valor is None:
        return None
    texto = str(valor).strip()
    match = REGEX_ANO.match(texto)
    return int(match.group(1)) if match and match.group(1) != "0000" else None


def tabela_ano(ano):
    return f"inform_27_{ano}"


def criar_tabela(con, ano):
    tabela = tabela_ano(ano)
    colunas_sql = ", ".join(f'"{c}" TEXT' for c in COLUNAS)

    con.execute(f'''
        CREATE TABLE IF NOT EXISTS "{tabela}" (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            {colunas_sql},
            "{COLUNA_ORIGEM}" TEXT
        )
    ''')

    con.execute(
        f'CREATE UNIQUE INDEX IF NOT EXISTS "idx_{tabela}" '
        f'ON "{tabela}" ("CODEDT", "FENTREGA", "CODEUT")'
    )
    return tabela


def inserir_linhas(con, ano, linhas):
    if not linhas:
        return 0, 0

    tabela = criar_tabela(con, ano)
    colunas_sql = ", ".join(f'"{c}"' for c in COLUNAS)
    placeholders = ", ".join("?" for _ in range(len(COLUNAS) + 1))

    sql = f'INSERT OR IGNORE INTO "{tabela}" ({colunas_sql}, "{COLUNA_ORIGEM}") VALUES ({placeholders})'

    antes = con.total_changes
    con.executemany(sql, linhas)
    inseridos = con.total_changes - antes

    return inseridos, len(linhas) - inseridos


# ==========================
# 1. Processamento ficheiro a ficheiro
# ==========================
ficheiros = sorted(PASTA_FICHEIROS.rglob("*.csv"))

if not ficheiros:
    raise FileNotFoundError(f"Nenhum ficheiro .csv em: {PASTA_FICHEIROS}")

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA synchronous = OFF")
con.execute("PRAGMA journal_mode = WAL")

totais = {
    "ficheiros": 0,
    "linhas": 0,
    "inseridos": {},
    "duplicados": 0,
    "sem_data": 0,
    "erros": 0
}

for numero, caminho in enumerate(ficheiros, 1):
    try:
        # ==========================
        # 2. Ler ficheiro CSV
        # ==========================
        with open(caminho, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            cabecalho = next(reader)
            linhas = list(reader)

        if COLUNA_DATA not in cabecalho:
            print(
                f"[{numero}/{len(ficheiros)}] "
                f"{caminho.name} — sem {COLUNA_DATA}"
            )
            totais["erros"] += 1
            continue

        mapa_indices = {
            coluna: i
            for i, coluna in enumerate(cabecalho)
        }

        linhas_por_ano = {}
        sem_data = 0

        # ==========================
        # 3. Preparar linhas
        # ==========================
        for valores in linhas:
            indice_data = mapa_indices.get(COLUNA_DATA)
            valor_data = (
                valores[indice_data]
                if indice_data is not None and indice_data < len(valores)
                else None
            )

            ano = obter_ano(valor_data)

            if ano is None:
                sem_data += 1
                continue

            linha = []
            for coluna in COLUNAS:
                indice = mapa_indices.get(coluna)
                valor = (
                    valores[indice]
                    if indice is not None and indice < len(valores)
                    else None
                )
                linha.append(valor)

            linha.append(caminho.name)
            linhas_por_ano.setdefault(ano, []).append(linha)

        # ==========================
        # 4. Inserir dados
        # ==========================
        novos_ficheiro = 0
        duplicados_ficheiro = 0

        for ano, batch in sorted(linhas_por_ano.items()):
            inseridos, duplicados = inserir_linhas(
                con,
                ano,
                batch
            )
            totais["inseridos"][ano] = (
                totais["inseridos"].get(ano, 0) + inseridos
            )
            novos_ficheiro += inseridos
            duplicados_ficheiro += duplicados

        # ==========================
        # 5. Confirmar ficheiro
        # ==========================
        con.commit()

        totais["ficheiros"] += 1
        totais["linhas"] += len(linhas)
        totais["duplicados"] += duplicados_ficheiro
        totais["sem_data"] += sem_data

        print(
            f"[{numero}/{len(ficheiros)}] "
            f"{caminho.name} | "
            f"+{novos_ficheiro:,} novos | "
            f"{duplicados_ficheiro:,} duplicados | "
            f"{sem_data:,} sem data"
        )

    except Exception as erro:
        # ==========================
        # 6. Reverter apenas ficheiro
        # ==========================
        con.rollback()
        totais["erros"] += 1
        print(
            f"[{numero}/{len(ficheiros)}] "
            f"{caminho.name} — ERRO: {erro}"
        )

# ==========================
# 7. Fechar ligação
# ==========================
con.close()

# ==========================
# 8. Resumo final
# ==========================
print("\n--- RESUMO ---")
print(
    f"Ficheiros: {totais['ficheiros']} | "
    f"Linhas lidas: {totais['linhas']:,} | "
    f"Duplicadas: {totais['duplicados']:,} | "
    f"Sem data: {totais['sem_data']:,} | "
    f"Erros: {totais['erros']}"
)
for ano, qtd in sorted(totais["inseridos"].items()):
    print(f"Linhas novas em {ano}: {qtd:,}")

[1/1] SAL_DAT027.csv | +23,459 novos | 172,950 duplicados | 31 sem data

--- RESUMO ---
Ficheiros: 1 | Linhas lidas: 196,440 | Duplicadas: 172,950 | Sem data: 31 | Erros: 0
Linhas novas em 2026: 23,459
